# CSC 349A Numerical Analysis - Lecture 13 - Gaussian Elimination with Partial Pivoting
The Naive Gaussian elimination algorithm will fail if any of the pivots $a_{11}, a_{22}^{(1)},a_{33}^{(2)}, ...$ is equal to $0$. 
Mathematically, it works provided this does not occur. Algorithmically, it breaks down when the pivots are even close to $0$ because 
of floating-point arithmetic. The problem occurs in the multiplier, it becomes far larger than the other entries

### Example 1 
Consider the $n=2$ linear system with augmented matrix with $k=4$, $b=10$, rounding, floating-point arithmetic.
$$
\left[
\begin{array}{rr|r}
	0.003	&	59.14	&	59.17 \\
	5.291		&	-6.13	&	46.78
\end{array}
\right]
$$

Note the the exact solution to this system is $x=\begin{bmatrix}10 \\ 1 \end{bmatrix}$. The multiplier is

$$
m_{21}=fl(a_{21}/a_{11})=fl(5.291/0.003)=fl(1763.666...)=1764
$$

Thus, forward elimination gives

\begin{align}
a_{22}&=fl(-6.13-fl(1764*59.14)) \\
&=fl(-6.13-fl(104322.96)) \\
&=fl(-6.13-104300) \\
&=fl(-104306.13) \\
&=-104300
\end{align}

and

\begin{align*}
b_2&=fl(46.78-fl(1764*59.17)) \\
&=fl(46.78-fl(104375.88)) \\
&=fl(46.78-104400) \\
&=fl(-104353.22) \\
&=-104400
\end{align*}

The new augmented matirix is

$$
\left[
\begin{array}{rr|r}
	0.003	&	59.14	&	59.17 \\
	0		&	-104300 &	 -104400
\end{array}
\right]
$$

Finally, we do back substitution giving

\begin{align}
x_2&=fl(-104400/-104300)=fl(1.000958...)=1.001 \\
x_1&=fl\left(\frac{b_1-a_{12}x_2}{a_{11}}\right) \\
&=fl\left(\frac{fl(59.17-fl(59.14*1.001))}{0.003}\right) \\
&=fl\left(\frac{fl(59.17-59.20)}{0.003}\right) \\
&= fl\left(\frac{-0.03}{0.003}\right) \\
&= -10
\end{align}

Therefore, $\hat{x}=\begin{bmatrix}-10 \\ 1.001 \end{bmatrix}$.

**Analysis of the above Example:**
The relative error in $\hat{x}_1$ is $200\%$. The source of the extremely inaccurate computed solution 
$\hat{x}$ is the **large magnitude of the multiplier**. Here, 1764 is much larger than the rest of 
the numbers in the system. 

This number is large because the pivot, $a_{11}=0.003$, is much smaller than the other numbers in the system. 
Consequently, in the floating-point computations of $a_{22}^{(1)}$ and $b_2^{(1)}$, the numbers $-6.13$ and 
$46.78$ are so small they are lost. The **partial pivoting strategy** is designed to avoid the selection of 
small pivots.

## Partial Pivoting
At step $k$ of forward elimination, where $1 \leq k \leq n-1$, choose the pivot to be the **largest entry in absolute value**, from
$$
\begin{bmatrix}
	a_{kk} \\
	a_{k+1,k} \\
	a_{k+2,k} \\
	\vdots \\
	a_{n,k}
\end{bmatrix}
$$

If $a_{pk}$ is the largest (that is, $|a_{pk}|=\max_{k \leq i \leq n}|a_{ik}|$), then switch row 
$k$ with row $p$. Note that $|mult| \leq 1$ for all multipliers since the denominator is always the 
largest value. Note also that switching rows does not change the final solution. It is an elementary 
row operation of type 3.

<div style="background-color:rgba(0, 0, 0, 0.0470588); padding:10px 0;font-family:monospace;">

**Algorithm 1** pseudocode for partial pivoting

1. **for** $k=1$ to $n-1$ **do**
2. &nbsp;&nbsp;&nbsp;$p=k$
3. &nbsp;&nbsp;&nbsp;**for** $i=k+1$ to $n$ **do**
4. &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Find largest pivot
5. &nbsp;&nbsp;&nbsp;**end for**
6. &nbsp;&nbsp;&nbsp;**if** $p \neq k$ **then**
7. &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**for** $j=k$ to $n$ **do**
8. &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;swap $a_{k,j}$ and $a_{p,j}$
9. &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**end for**
10. &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;swap $b_k$ and $b_p$
11. &nbsp;&nbsp;&nbsp;**end if**
12. &nbsp;&nbsp;&nbsp;do forward elimination
13. **end for**
14. do back substitution
</div>

### Example 2 
Consider the $n=2$ linear system with augmented matrix with $k=4$, $b=10$, rounding, 
floating-point arithmetic using partial pivoting.
$$
\left[
\begin{array}{rr|r}
	0.003	&	59.14	&	59.17 \\
	5.291		&	-6.13	&	46.78
\end{array}
\right]
$$

Note the the exact solution to this system is $x=\begin{bmatrix}10 \\ 1 \end{bmatrix}$. Because $|5.291| > |0.003|$, 
we swap rows (equations) 1 and 2, thus
$$
\left[
\begin{array}{rr|r}
	5.291		&	-6.13	&	46.78 \\
	0.003	&	59.14	&	59.17
\end{array}
\right]
$$

The multiplier is

$$
m_{21}=fl(a_{21}/a_{11})=fl(0.003/5.291)=fl(0.000567)=0.000567
$$

Thus, forward elimination gives

\begin{align}
a_{22}&=fl(59.14-fl(0.000567*(-6.13))) \\
&=fl(59.14-fl(-0.003475713)) \\
&=fl(59.14+0.003476) \\
&=fl(59.143476) \\
&=59.14
\end{align}

and

\begin{align}
b_2&=fl(59.17-fl(0.000567*46.78)) \\
&=fl(59.17-fl(0.02652426)) \\
&=fl(59.17-0.02652) \\
&=fl(59.14348) \\
&=59.14
\end{align}

The new augmented matirix is

$$
\left[
\begin{array}{rr|r}
	5.291		&	-6.13	&	46.78 \\
	0	&	59.14	&	59.14
\end{array}
\right]
$$

Finally, we do back substitution giving

\begin{align}
x_2&=fl(59.14/59.14)=1 \\
x_1&=fl\left(\frac{b_1-a_{12}x_2}{a_{11}}\right) \\
&=fl\left(\frac{fl(46.78-fl(-6.13*1))}{5.291}\right) \\
&=fl\left(\frac{fl(46.78+6.13)}{5.291}\right) \\
&= fl\left(\frac{52.91}{5.291}\right) \\
&= 10
\end{align}

Therefore, $\hat{x}=\begin{bmatrix} 10 \\ 1 \end{bmatrix}$ which is exactly correct.